# MLP Hyperparameter Optimization

Optuna searches MLP depth, width, activation, and learning rate for the infinite-domain inverse problem. Each completed trial is saved in a timestamped results directory with its models, metrics, and training history.

In [9]:
import os
import sys
from datetime import datetime

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns
import infinite
reload(plotting)
reload(pinns)
reload(infinite)
import numpy as np
import sympy as sp
from calflops import calculate_flops
import pandas as pd
import joblib
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from pinns import build_models
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison

set_seed(1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [10]:
import optuna

# Search space from the architecture-matching study.
MLP_SEARCH_SPACE = {
    "hidden_layers": [1, 2, 3],
    "hidden_units": [15, 90, 104],
    "activation": ["Sine", "Sigmoid", "Tanh"],
    "learning_rate": [1e-4, 1e-3, 1e-2],
}

ACTIVATIONS = {
    "Sine": lambda: Sine(),
    "Sigmoid": nn.Sigmoid,
    "Tanh": nn.Tanh,
}


class Sine(nn.Module):
    def forward(self, x):
        return torch.sin(x)


N_TRIALS = 50
ADAM_ITERS = 2000
LBFGS_ITERS = 2000

# One directory contains the CSV summary and all saved trial models.
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
results_dir = f"results_mlp_optuna_{timestamp}"
os.makedirs(results_dir, exist_ok=True)

print(f"Results will be saved to: {results_dir}")
print(f"Optuna trials: {N_TRIALS}")

Results will be saved to: results_mlp_optuna_2026-09-15_10-21-28
Optuna trials: 50


## Objective Function

Each trial trains one MLP configuration and minimizes the mean of the global relative errors for `u` and `k`.

In [11]:
def objective(trial):
    """Run one MLP training configuration and return mean global error."""
    config = {
        "hidden_layers": trial.suggest_categorical(
            "hidden_layers",
            MLP_SEARCH_SPACE["hidden_layers"],
        ),
        "hidden_units": trial.suggest_categorical(
            "hidden_units",
            MLP_SEARCH_SPACE["hidden_units"],
        ),
        "activation": trial.suggest_categorical(
            "activation",
            MLP_SEARCH_SPACE["activation"],
        ),
        "learning_rate": trial.suggest_categorical(
            "learning_rate",
            MLP_SEARCH_SPACE["learning_rate"],
        ),
    }
    activation = ACTIVATIONS[config["activation"]]()

    print(
        f"\n--- Trial {trial.number}: "
        f"L={config['hidden_layers']}, "
        f"N={config['hidden_units']}, "
        f"activation={config['activation']}, "
        f"lr={config['learning_rate']:.0e} ---"
    )

    try:
        err_u, err_k, compute_time = run_experiment_inf(
            model_type="MLP",
            hidden_layers=config["hidden_layers"],
            hidden_units=config["hidden_units"],
            activation=activation,
            adam_lr=config["learning_rate"],
            device=device,
            adam_iters=ADAM_ITERS,
            lbfgs_iters=LBFGS_ITERS,
            results_dir=results_dir,
        )
    except Exception as error:
        print(f"Trial {trial.number} failed: {error}")
        raise optuna.exceptions.TrialPruned()

    mean_global_error = 0.5 * (err_u + err_k)
    trial.set_user_attr("err_u", float(err_u))
    trial.set_user_attr("err_k", float(err_k))
    trial.set_user_attr("compute_time_sec", float(compute_time))

    print(
        f"Success! Time: {compute_time:.2f}s | "
        f"Err U: {err_u:.3e} | Err K: {err_k:.3e} | "
        f"Mean error: {mean_global_error:.3e}"
    )
    return mean_global_error

## Run Optimization

In [12]:
sampler = optuna.samplers.TPESampler(seed=1)
study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    study_name=f"mlp_infinite_domain_{timestamp}",
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    catch=(RuntimeError, ValueError),
)

print("\n========================================")
print("BEST MLP CONFIGURATION")
print("========================================")
print(f"Mean global error: {study.best_value:.6e}")
print("Parameters:")
for name, value in study.best_params.items():
    print(f"  {name}: {value}")

[I 2026-09-15 10:21:28,728] A new study created in memory with name: mlp_infinite_domain_2026-09-15_10-21-28



--- Trial 0: L=2, N=15, activation=Tanh, lr=1e-02 ---


[I 2026-09-15 10:22:31,041] Trial 0 finished with value: 0.06401409904405604 and parameters: {'hidden_layers': 2, 'hidden_units': 15, 'activation': 'Tanh', 'learning_rate': 0.01}. Best is trial 0 with value: 0.06401409904405604.



[MLP] L=2, N=15 | Params: 1,082 | Mean Err: 6.401e-02 | Saved to 'results_mlp_optuna_2026-09-15_10-21-28/'.
Success! Time: 62.31s | Err U: 2.032e-02 | Err K: 1.077e-01 | Mean error: 6.401e-02

--- Trial 1: L=2, N=15, activation=Tanh, lr=1e-04 ---


[I 2026-09-15 10:23:35,161] Trial 1 finished with value: 0.08301306757146287 and parameters: {'hidden_layers': 2, 'hidden_units': 15, 'activation': 'Tanh', 'learning_rate': 0.0001}. Best is trial 0 with value: 0.06401409904405604.


...


[MLP] L=3, N=90 | Params: 49,862 | Mean Err: 7.307e-04 | Saved to 'results_mlp_optuna_2026-09-15_10-21-28/'.
Success! Time: 75.47s | Err U: 1.110e-03 | Err K: 3.509e-04 | Mean error: 7.307e-04

--- Trial 48: L=1, N=90, activation=Tanh, lr=1e-03 ---


[I 2026-09-15 11:20:34,431] Trial 48 finished with value: 0.01083173234054539 and parameters: {'hidden_layers': 1, 'hidden_units': 90, 'activation': 'Tanh', 'learning_rate': 0.001}. Best is trial 24 with value: 0.000730671142914815.



[MLP] L=1, N=90 | Params: 17,102 | Mean Err: 1.083e-02 | Saved to 'results_mlp_optuna_2026-09-15_10-21-28/'.
Success! Time: 66.16s | Err U: 1.814e-02 | Err K: 3.520e-03 | Mean error: 1.083e-02

--- Trial 49: L=1, N=90, activation=Sine, lr=1e-02 ---


[I 2026-09-15 11:21:39,179] Trial 49 finished with value: 0.9774561515461136 and parameters: {'hidden_layers': 1, 'hidden_units': 90, 'activation': 'Sine', 'learning_rate': 0.01}. Best is trial 24 with value: 0.000730671142914815.



[MLP] L=1, N=90 | Params: 17,102 | Mean Err: 9.775e-01 | Saved to 'results_mlp_optuna_2026-09-15_10-21-28/'.
Success! Time: 64.73s | Err U: 5.908e-02 | Err K: 1.896e+00 | Mean error: 9.775e-01

BEST MLP CONFIGURATION
Mean global error: 7.306711e-04
Parameters:
  hidden_layers: 3
  hidden_units: 90
  activation: Sine
  learning_rate: 0.001


## Save Optimization Results

In [13]:
# Persist the complete study and a tabular summary for later analysis.
data_dir = os.path.join(results_dir, "data")
os.makedirs(data_dir, exist_ok=True)

joblib.dump(study, os.path.join(data_dir, "study.pkl"))
joblib.dump(study, os.path.join(data_dir, f"study_{timestamp}.pkl"))

study_df = study.trials_dataframe()
study_csv_path = os.path.join(data_dir, "study.csv")
study_df.to_csv(study_csv_path, index=False)

completed_df = study_df[
    study_df["state"].eq("COMPLETE")
].sort_values(by="value", ascending=True)
completed_csv_path = os.path.join(data_dir, "study_completed_sorted.csv")
completed_df.to_csv(completed_csv_path, index=False)

print(f"Saved study to: {data_dir}")
print(f"Saved trial summary to: {study_csv_path}")
print(f"Saved sorted completed trials to: {completed_csv_path}")

Saved study to: results_mlp_optuna_2026-09-15_10-21-28/data
Saved trial summary to: results_mlp_optuna_2026-09-15_10-21-28/data/study.csv
Saved sorted completed trials to: results_mlp_optuna_2026-09-15_10-21-28/data/study_completed_sorted.csv
